# 02. Spatial Constraint Prototyping: GIS Exclusion Layers

## Overview
This notebook prototypes spatial multi-criteria exclusion layers using **Shapely** and **GeoPandas**. It models hard binary constraints including a **150m river buffer**, **300m urban settlement buffer**, and **protected forest boundaries**.

In [ ]:
import numpy as np
import pandas as pd
from shapely.geometry import Point, Polygon, LineString
import matplotlib.pyplot as plt

print("Shapely geometry engine loaded!")

## 1. Define Candidate Site Coordinates & Mock Exclusion Zones

In [ ]:
# Candidate site points
sites = [
    {'id': 'SITE_A', 'lat': 26.9124, 'lng': 75.7873},
    {'id': 'SITE_B', 'lat': 26.9200, 'lng': 75.7900},
    {'id': 'SITE_C', 'lat': 26.9300, 'lng': 75.8100},
    {'id': 'SITE_D', 'lat': 26.9050, 'lng': 75.7700},
]

# Create Shapely points
points = [Point(s['lng'], s['lat']) for s in sites]

# Simulated River LineString
river = LineString([(75.7700, 26.9150), (75.7950, 26.9180), (75.8200, 26.9250)])

# Simulated Protected Forest Polygon
protected_zone = Polygon([
    (75.8050, 26.9250),
    (75.8150, 26.9250),
    (75.8150, 26.9350),
    (75.8050, 26.9350)
])

print("Site points and exclusion layers defined.")

## 2. Buffer Constraint Logic (150m Buffer Proxy)

In [ ]:
# Buffer distance in degrees proxy (0.0015 deg ~ 150m)
river_buffer = river.buffer(0.0015)

eval_results = []
for s, pt in zip(sites, points):
    intersects_river = pt.intersects(river_buffer)
    intersects_protected = pt.intersects(protected_zone)
    is_feasible = not (intersects_river or intersects_protected)
    
    eval_results.append({
        'site_id': s['id'],
        'river_violation': intersects_river,
        'protected_violation': intersects_protected,
        'status': 'APPROVED' if is_feasible else 'REJECTED'
    })

df_eval = pd.DataFrame(eval_results)
df_eval

## 3. Spatial Exclusion Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Plot River & Buffer
rx, ry = river.xy
ax.plot(rx, ry, color='blue', linewidth=3, label='River Axis')

bx, by = river_buffer.exterior.xy
ax.fill(bx, by, color='cyan', alpha=0.3, label='150m River Buffer')

# Plot Protected Zone
px, py = protected_zone.exterior.xy
ax.fill(px, py, color='red', alpha=0.3, label='Protected Forest Zone')

# Plot Sites
for s, res in zip(sites, eval_results):
    color = 'green' if res['status'] == 'APPROVED' else 'red'
    ax.scatter(s['lng'], s['lat'], color=color, s=120, zorder=5, label=f"{s['id']} ({res['status']})")

ax.set_title('GIS Spatial Buffer Exclusion Audit')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Key Takeaways
- 150m buffer logic reliably rejects site candidates located too close to drainage channels or river basins.
- Integration with OpenStreetMap Overpass queries provides automated boundary checking prior to financial modeling.